In [1]:
import pandas as pd
from src.metrics import evaluate_regression
from src.models.custom_mlp import build_custom_mlp
import time

In [2]:
data = pd.read_csv("/home/vo/project_price/dataset/staging/data_sau_clean.csv")

print("Shape:", data.shape)

Shape: (117322, 13)


In [3]:
from src.feature_engineering.build_feature import build_features

features = build_features(
    data
)

X_train = features["X_train"]
X_test  = features["X_test"]
y_train = features["y_train"]
y_test  = features["y_test"]

In [4]:
X_train.head(5)

,area,bedrooms,bathrooms,bedrooms_is_missing,bathrooms_is_missing,days_on_market,city_An Giang,city_Bà Rịa Vũng Tàu,city_Bình Dương,city_Bình Phước,...,property_type_Shophouse,property_type_Trang trại,property_type_Đất nền,transaction_type_Bán,legal_status_Dang_Cho_So,legal_status_Hop_Dong,legal_status_Other_Unknown,legal_status_So_Do_So_Hong,legal_status_Unknown,legal_status_Vi_Bang
114449,0.024729,-0.136336,-0.061787,-0.843450,-0.905125,-0.407888,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
36464,0.214354,0.607025,0.743728,-0.843450,-0.905125,-0.224585,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
74776,-0.414856,-0.384123,-0.330292,-0.843450,-0.905125,-0.206255,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
36504,0.248832,-0.136336,-0.061787,-0.843450,-0.905125,-0.334567,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
74771,0.886662,0.111451,0.206718,1.185607,1.104820,-0.206255,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


In [5]:
import numpy as np
from sklearn.model_selection import train_test_split

# Chia TRAIN → TRAIN + VAL
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

# Ép kiểu AN TOÀN
X_train = np.asarray(X_train)
y_train = np.asarray(y_train)

X_val = np.asarray(X_val)
y_val = np.asarray(y_val)

X_test = np.asarray(X_test)
y_test = np.asarray(y_test)

print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)
print("Test :", X_test.shape, y_test.shape)



Train: (75085, 534) (75085,)
Val  : (18772, 534) (18772,)
Test : (23465, 534) (23465,)


In [6]:
print(type(X_train))
print(type(y_train))
print(type(X_val))
print(type(y_val))
print(type(X_test))
print(type(y_test))

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [7]:
print(X_train.shape)
print(y_train.shape)
print(X_val.shape)
print(y_val.shape)
print(X_test.shape)
print(y_test.shape)

(75085, 534)
(75085,)
(18772, 534)
(18772,)
(23465, 534)
(23465,)


In [8]:
y_train = y_train.reshape(-1, 1)
y_val   = y_val.reshape(-1, 1)
y_test  = y_test.reshape(-1, 1)


In [9]:
results = {}
mlp = build_custom_mlp(X_train.shape[1])
start = time.time()
mlp.fit(X_train, y_train, X_val, y_val)
train_time = time.time() - start
y_pred = mlp.predict(X_test)
results["CustomMLP"] = { **evaluate_regression(y_test, y_pred),
    "train_time": train_time}


Epoch 093 | Train 0.1350 | Val 0.2675
Early stopping


In [10]:
for model, metrics in results.items():
    print(f"\n{model}")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")


CustomMLP
  MAE: 0.2542
  RMSE: 0.4500
  R2: 0.7966
  train_time: 388.6842


In [17]:
results1  = {}
y_train_pred = mlp.predict(X_train)
y_val_pred   = mlp.predict(X_val)


# ================== Evaluate ==================
results1["CustomMLP"] = {
    "train": evaluate_regression(y_train, y_train_pred),
    "val":   evaluate_regression(y_val, y_val_pred),
}


In [20]:
for model, metrics in results1.items():
    print(f"\n{model}")

    for split in ["train", "val"]:
        print(f"  [{split.upper()}]")
        for m, v in metrics[split].items():
            print(f"    {m}: {v:.4f}")





CustomMLP
  [TRAIN]
    MAE: 0.2588
    RMSE: 0.4201
    R2: 0.8209
  [VAL]
    MAE: 0.2738
    RMSE: 0.5143
    R2: 0.7503


In [13]:
from src.models.customMLP_Wide import build_custom_mlp_wide
from src.models.customMLP_Smooth import build_custom_mlp_smooth

In [14]:
results1 = {}
mlp1 = build_custom_mlp_wide(X_train.shape[1])
start = time.time()
mlp1.fit(X_train, y_train, X_val, y_val)
train_time = time.time() - start
y_pred = mlp.predict(X_test)
results1["CustomMLP"] = { **evaluate_regression(y_test, y_pred),
    "train_time": train_time}

Epoch 001 | Train 0.2683 | Val 0.3344
Epoch 002 | Train 0.2479 | Val 0.3172
Epoch 003 | Train 0.2294 | Val 0.3007
Epoch 004 | Train 0.2180 | Val 0.2916
Epoch 005 | Train 0.2093 | Val 0.2867
Epoch 006 | Train 0.2041 | Val 0.2840
Epoch 007 | Train 0.1989 | Val 0.2814
Epoch 008 | Train 0.1941 | Val 0.2788
Epoch 009 | Train 0.1888 | Val 0.2758
Epoch 010 | Train 0.1857 | Val 0.2741
Epoch 011 | Train 0.1819 | Val 0.2719
Epoch 012 | Train 0.1803 | Val 0.2718
Epoch 013 | Train 0.1767 | Val 0.2699
Epoch 014 | Train 0.1752 | Val 0.2698
Epoch 015 | Train 0.1754 | Val 0.2717
Epoch 016 | Train 0.1719 | Val 0.2689
Epoch 017 | Train 0.1688 | Val 0.2674
Epoch 018 | Train 0.1662 | Val 0.2656
Epoch 019 | Train 0.1670 | Val 0.2681
Epoch 020 | Train 0.1656 | Val 0.2663
Epoch 021 | Train 0.1678 | Val 0.2723
Epoch 022 | Train 0.1641 | Val 0.2686
Epoch 023 | Train 0.1632 | Val 0.2676
Epoch 024 | Train 0.1618 | Val 0.2656
Epoch 025 | Train 0.1614 | Val 0.2689
Epoch 026 | Train 0.1606 | Val 0.2666
Reduce LR
Ep

In [15]:
for model, metrics in results1.items():
    print(f"\n{model}")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")


CustomMLP
  MAE: 0.2557
  RMSE: 0.4498
  R2: 0.7969
  train_time: 394.1406


In [69]:
print(results1)

{'CustomMLP': {'MAE': 0.2556718628755607, 'RMSE': np.float64(0.44977138503012837), 'R2': 0.7968884919677361, 'train_time': 531.6429853439331}}


In [41]:
results2  = {}
mlp1 = build_custom_mlp_smooth(X_train.shape[1])
start = time.time()
mlp1.fit(X_train, y_train, X_val, y_val)
train_time = time.time() - start
y_pred = mlp.predict(X_test)
results["CustomMLP"] = { **evaluate_regression(y_test, y_pred),
    "train_time": train_time}

Epoch 001 | Train 0.3940 | Val 0.4573
Epoch 002 | Train 0.3605 | Val 0.4240
Epoch 003 | Train 0.3457 | Val 0.4089
Epoch 004 | Train 0.3319 | Val 0.3948
Epoch 005 | Train 0.3232 | Val 0.3863
Epoch 006 | Train 0.3164 | Val 0.3788
Epoch 007 | Train 0.3088 | Val 0.3721
Epoch 008 | Train 0.3054 | Val 0.3694
Epoch 009 | Train 0.2983 | Val 0.3618
Epoch 010 | Train 0.2944 | Val 0.3584
Epoch 011 | Train 0.2899 | Val 0.3536
Epoch 012 | Train 0.2870 | Val 0.3509
Epoch 013 | Train 0.2829 | Val 0.3462
Epoch 014 | Train 0.2808 | Val 0.3445
Epoch 015 | Train 0.2781 | Val 0.3425
Epoch 016 | Train 0.2753 | Val 0.3398
Epoch 017 | Train 0.2737 | Val 0.3384
Epoch 018 | Train 0.2709 | Val 0.3348
Epoch 019 | Train 0.2723 | Val 0.3374
Epoch 020 | Train 0.2698 | Val 0.3334
Epoch 021 | Train 0.2665 | Val 0.3309
Epoch 022 | Train 0.2653 | Val 0.3302
Epoch 023 | Train 0.2669 | Val 0.3316
Epoch 024 | Train 0.2634 | Val 0.3278
Epoch 025 | Train 0.2639 | Val 0.3278
Epoch 026 | Train 0.2638 | Val 0.3291
Epoch 027 | 